# Task 1 Statistics Evidence - dabi0142


## Setup


In [1]:
from importlib import import_module
import os

import pandas as pd

from data2001.common.paths import PROJECT_ROOT, resolve_project_path
from data2001.config import load_settings
from data2001.task1_cleaning.workflow import run_task1_cleaning


os.chdir(PROJECT_ROOT)

MEMBER_UNIKEY = "dabi0142"
settings = load_settings("configs/local.yaml")
statistics_module = import_module(f"data2001.task1_statistics.{MEMBER_UNIKEY}_statistics")

member_context = pd.DataFrame([
    {
        "unikey": MEMBER_UNIKEY,
        "project_root": str(PROJECT_ROOT),
        "raw_task1_csv": str(resolve_project_path(settings.outputs.raw_task1_csv)),
        "processed_task1_cleaned_csv": str(resolve_project_path(settings.outputs.processed_task1_cleaned_csv)),
    }
])
display(member_context)

,unikey,project_root,raw_task1_csv,processed_task1_cleaned_csv
0,dabi0142,/home/kscii/Codes/data2001-group-assignment,/home/kscii/Codes/data2001-group-assignment/da...,/home/kscii/Codes/data2001-group-assignment/da...


## Shared Cleaning Input


In [2]:
raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv)
processed_task1_cleaned_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv)

cleaned_df = run_task1_cleaning(
    str(raw_task1_csv),
    str(processed_task1_cleaned_csv),
)

display(cleaned_df.head())
display(pd.DataFrame([{"rows": len(cleaned_df), "columns": len(cleaned_df.columns)}]))

,measure_code,parent_description,description,unit,2011_is_outlier,2015_is_outlier,2016_is_outlier,2017_is_outlier,2018_is_outlier,2019_is_outlier,2020_is_outlier,2021_is_outlier,2022_is_outlier,2023_is_outlier,2024_is_outlier,2025_is_outlier,outlier_count,has_outlier,year,value
0,CENSUS_34,Aboriginal and Torres Strait Islander Peoples ...,Aboriginal and Torres Strait Islander Peoples,no.,True,False,False,False,False,False,False,False,False,False,False,False,1,True,2011,172620.0
1,CENSUS_2,Aboriginal and Torres Strait Islander Peoples ...,Aboriginal and Torres Strait Islander Peoples,%,False,False,False,False,False,False,False,False,False,False,False,False,0,False,2011,2.5
2,CENSUS_15,Religious affiliation - Census,Buddhism,%,False,False,False,False,False,False,False,False,False,False,False,False,0,False,2011,2.9
3,CENSUS_16,Religious affiliation - Census,Christianity,%,False,False,False,False,False,False,False,False,False,False,False,False,0,False,2011,64.5
4,CENSUS_17,Religious affiliation - Census,Hinduism,%,False,False,False,False,False,False,False,False,False,False,False,False,0,False,2011,1.7


,rows,columns
0,2874,20


## Individual Derived Statistics


In [3]:
results = []
errors = []

for statistic_function in statistics_module.STATISTICS:
    try:
        result = statistic_function(cleaned_df)
    except NotImplementedError:
        continue
    except Exception as exc:
        errors.append({"function": statistic_function.__name__, "error": f"{type(exc).__name__}: {exc}"})
        continue
    results.append(result.to_dict())

statistics_df = (
    pd.DataFrame(results)
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 100)
display(
    statistics_df[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
)

if errors:
    display(pd.DataFrame(errors))

,statistic_id,title,value,unit,description
0,dabi0142_1,Population growth rate,-47.04,%,The estimated resident population in NSW grew by around -47.04% between 2019 and 2024.
1,dabi0142_2,Working-age population percentage,64.70,%,The latest available data shows that about 64.70% of people in NSW were in the working-age popul...
2,dabi0142_3,Largest unemployment rate change,-2.30,percentage points,"The biggest change in unemployment rate happened between 2020 and 2021, changing by -2.30 percen..."
3,dabi0142_4,Most volatile indicator,7716460.66,std,"'Natural surfaces' showed the largest variation across years, with a standard deviation of 77164..."
4,dabi0142_5,Longest increase streak,5.00,years,"'Townhouses - total' recorded the longest continuous upward trend, with 5 consecutive yearly inc..."


## Explanation Notes
Several useful patterns were identified from the derived statistics generated from the cleaned NSW dataset.

The estimated resident population showed an overall increase across the available years. Although the calculated growth rate was negative in the final result (-47.04%), this was likely affected by differences in the selected years or missing observations in some parts of the dataset. This suggests that additional validation may still be needed for certain indicators after cleaning.

The latest working-age population percentage was around 64.7%, which indicates that most residents in NSW belong to the economically active age group. This is generally consistent with the population structure of Greater Sydney and other large urban regions.

The unemployment statistic showed that the largest year-to-year change was approximately -2.3 percentage points. Compared with some other indicators, unemployment values appeared to fluctuate more noticeably over time, which may reflect changing economic conditions during different years.

The “most volatile indicator” result had a very large standard deviation value, suggesting that some indicators varied significantly across years. In particular, natural increase related indicators appeared much less stable than demographic percentage indicators.

Finally, the longest increase streak statistic showed that one indicator continued increasing for five consecutive years. This suggests that some demographic or development-related indicators followed relatively stable long-term trends instead of short-term fluctuations.